In [15]:
# create 2 tools add and multiply
# create normal prompt and AI would answer and not use the tool
# Provide system prompt if add or multiply, use the tool then model (llm) will use the tool

from langchain_ollama import ChatOllama

model = ChatOllama(model="llama3.2:latest")
print('Model is ready')

Model is ready


In [16]:
from typing import TypedDict, Annotated
from langchain_core.messages import BaseMessage, AnyMessage
from langgraph.graph.message import add_messages

class ReActState(TypedDict):
    messages: Annotated[
        list[AnyMessage],
        add_messages
    ]

print('ReAct State is ready')

ReAct State is ready


In [18]:
from langchain_core.tools import tool
from langgraph.prebuilt import ToolNode

# Addition node
@tool
def plus(v1: int, v2: int):
    """Adds two numbers"""
    return v1 + v2

# MUltiplication node
@tool
def cross(v1: int, v2: int):
    """Multiplies two numnbers"""
    return v1 * v2

TOOLS = [plus, cross]
tool_node = ToolNode(TOOLS)

SYSTEM_PROMPT = """You're a helpful assistant that can perform calculations using the tools provided.
If anyone asks to add two numbers then please suggest the add tool and for multiplication please suggest the multiply tool."""

llm_with_tools = model.bind_tools(TOOLS)
def generate_answer(state:ReActState):
    messages = state['messages']
    response = llm_with_tools.invoke([SYSTEM_PROMPT] + messages)
    return {"messages": [response] }

def should_continue(state: ReActState):
    last_message = state["messages"][-1]
    if last_message.tool_calls:
        return "out_tools"
    return "out_end"

print('Nodes are ready')

Nodes are ready


In [21]:
from langgraph.graph import StateGraph, START, END

builder = StateGraph(ReActState)

# Nodes
builder.add_node("generate_answer_node", generate_answer)
builder.add_node("tools_node", tool_node)

# Edges
builder.add_edge(START, "generate_answer_node")
builder.add_conditional_edges(
    "generate_answer_node",
    should_continue,
    {
        "out_tools": "tools_node",
        "out_end": END
    }
)
builder.add_edge("tools_node", "generate_answer_node")

# graph complie
graph = builder.compile()
print('Graph is ready')
# graph

Graph is ready


In [20]:
result = graph.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "What is the value of 10 + 20 and then multiplied by 5?"
            }
        ]
    }
)
result

{'messages': [HumanMessage(content='What is the value of 10 + 20 and then multiplied by 5?', additional_kwargs={}, response_metadata={}, id='c4568f9c-e4fa-457b-b2ae-a4760be21950'),
  AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'llama3.2:latest', 'created_at': '2026-11-10T15:06:49.7375592Z', 'done': True, 'done_reason': 'stop', 'total_duration': 19684531900, 'load_duration': 4902291700, 'prompt_eval_count': 262, 'prompt_eval_duration': 8894776000, 'eval_count': 48, 'eval_duration': 5881232000, 'logprobs': None, 'model_name': 'llama3.2:latest', 'model_provider': 'ollama'}, id='lc_run--01a1c5ff-1123-715c-a626-e3aaf2f19b52-0', tool_calls=[{'name': 'plus', 'args': {'v1': '10', 'v2': '20'}, 'id': 'a0e17bbd-3037-4e44-a538-013ca2d864dc', 'type': 'tool_call'}, {'name': 'cross', 'args': {'v1': 'result', 'v2': '5'}, 'id': '12da2e47-cb8f-49ca-9feb-0bb8a9678488', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 262, 'output_tokens': 48, 'tota

In [22]:
response = graph.invoke({
    "question": "What is the result of 2 plus 3, finally multiplied by 5"
})
response

{'messages': [AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'llama3.2:latest', 'created_at': '2026-11-10T15:07:54.8533751Z', 'done': True, 'done_reason': 'stop', 'total_duration': 3060830900, 'load_duration': 10323600, 'prompt_eval_count': 245, 'prompt_eval_duration': 400388000, 'eval_count': 24, 'eval_duration': 2622454000, 'logprobs': None, 'model_name': 'llama3.2:latest', 'model_provider': 'ollama'}, id='lc_run--01a1c600-506e-76e6-95c9-3fb7a3974c22-0', tool_calls=[{'name': 'plus', 'args': {'v1': '2', 'v2': '3'}, 'id': 'ca826ea4-8d7b-4ffc-a1f1-f7f0aa2b75e9', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 245, 'output_tokens': 24, 'total_tokens': 269}),
  ToolMessage(content='5', name='plus', id='a3fcc9ba-a4a5-41f3-9407-09c027fe1dbc', tool_call_id='ca826ea4-8d7b-4ffc-a1f1-f7f0aa2b75e9'),
  AIMessage(content='The output of the tool suggests that the answer to the addition question is 5. So, the answer to the original user questio